# Module 14 — Object-Oriented Programming

**Part III: Intermediate** · Level: Intermediate → Advanced · Prerequisites: Modules 8, 9, 10

## What you'll learn

- Classes, instances, attributes, and methods
- Special ("dunder") methods: `__init__`, `__repr__`, `__eq__`, `__len__`, `__call__`
- Inheritance, `super()`, and polymorphism
- Encapsulation, properties, and class/static methods
- Composition and the template-method pattern

---

## 14.1 Classes and instances

A **class** bundles data (attributes) and behaviour (methods) into a reusable template. An **instance** is one concrete object built from it.

In [ ]:
class Dog:
    species = "Canis familiaris"          # class attribute: shared by all instances

    def __init__(self, name, age):        # initialiser, runs on creation
        self.name = name                  # instance attributes
        self.age = age

    def bark(self):
        return f"{self.name} says woof"

rex = Dog("Rex", 3)
fido = Dog("Fido", 5)
print(rex.bark(), "|", fido.name, "|", Dog.species)
print(type(rex), isinstance(rex, Dog))

`self` is the instance the method was called on; Python passes it automatically.

## 14.2 Special methods

Methods with double underscores let your objects work with Python's built-in syntax.

| Method | Triggered by |
|---|---|
| `__init__` | `Class(...)` |
| `__repr__` | `repr(obj)`, shown in the REPL and debugger |
| `__str__` | `str(obj)`, `print(obj)` |
| `__eq__` | `a == b` |
| `__len__` | `len(obj)` |
| `__call__` | `obj(...)` |
| `__lt__` etc. | `<`, `sorted()` |
| `__enter__` / `__exit__` | `with` (Module 16) |
| `__iter__` / `__next__` | `for` loops (Module 16) |

In [ ]:
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __repr__(self):
        return f"Money(cents={self.cents!r})"       # unambiguous, ideally eval-able

    def __str__(self):
        return f"${self.cents / 100:,.2f}"          # friendly for users

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.cents == other.cents

    def __hash__(self):                             # define when you define __eq__
        return hash(self.cents)

    def __add__(self, other):
        return Money(self.cents + other.cents)

    def __lt__(self, other):
        return self.cents < other.cents

a, b = Money(1999), Money(501)
print(a, repr(a), a + b, a == Money(1999), a < b)
print(sorted([a, b]))
print(len({a, Money(1999)}))        # 1 -- equal objects collapse in a set

### Callable objects and `__len__`

In [ ]:
class Greeter:
    def __init__(self, greeting):
        self.greeting = greeting

    def __call__(self, name):
        return f"{self.greeting}, {name}!"

hello = Greeter("Hello")
print(hello("Ada"), callable(hello))

class Playlist:
    def __init__(self, *songs):
        self.songs = list(songs)

    def __len__(self):
        return len(self.songs)

    def __repr__(self):
        return f"<Playlist n={len(self)}>"

print(len(Playlist("a", "b")), Playlist("a"))

## 14.3 Inheritance and `super()`

A **subclass** inherits attributes and methods from a **parent** class, and can extend or override them. `super()` calls the parent's version.

In [ ]:
class Shape:
    def __init__(self, name):
        self.name = name

    def area(self):
        raise NotImplementedError("subclasses must implement area()")

    def describe(self):
        return f"{self.name} with area {self.area():.2f}"

class Rectangle(Shape):
    def __init__(self, width, height):
        super().__init__("rectangle")      # run the parent initialiser
        self.width, self.height = width, height

    def area(self):
        return self.width * self.height

class Circle(Shape):
    def __init__(self, radius):
        super().__init__("circle")
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

print(Rectangle(3, 4).describe())
print(Circle(1).describe())
print(issubclass(Circle, Shape), isinstance(Circle(1), Shape))

In [ ]:
class Parent:
    def __init__(self):
        print("parent")

class Child(Parent):
    def __init__(self):
        super().__init__()
        print("child")

Child()

## 14.4 Polymorphism and duck typing

**Polymorphism**: different objects respond to the same interface in their own way. Python uses *duck typing* — if it has the right method, it works; no shared base class is required.

In [ ]:
class Cat:
    def speak(self):
        return "meow"

class Robot:
    def speak(self):
        return "beep"

for thing in [Cat(), Robot(), Dog("Rex", 3).bark]:
    print(thing.speak() if hasattr(thing, "speak") else thing())

shapes = [Rectangle(2, 3), Circle(2)]
print([round(s.area(), 1) for s in shapes])    # same call, different behaviour

## 14.5 Encapsulation and properties

Python has no true private attributes. A leading underscore (`_balance`) is a *convention* meaning "internal — please do not touch". A double underscore (`__x`) triggers name-mangling to avoid accidental clashes in subclasses.

A **property** lets you add validation behind what looks like a plain attribute:

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = 0
        self.balance = balance             # goes through the setter

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value

    def deposit(self, amount):
        self.balance += amount

acct = Account("sam", 100)
acct.deposit(50)
print(acct.balance)
try:
    acct.balance = -5
except ValueError as err:
    print("rejected:", err)

## 14.6 Class methods and static methods

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f):            # alternative constructor
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def is_valid(celsius):                  # no self/cls: just a namespaced function
        return celsius >= -273.15

    def __repr__(self):
        return f"Temperature({self.celsius:.1f}C)"

print(Temperature.from_fahrenheit(212), Temperature.is_valid(-300))

## 14.7 Composition vs inheritance

"Is-a" suggests inheritance (`Circle` is a `Shape`); "has-a" suggests **composition** (a `Car` has an `Engine`). Prefer composition when in doubt — it is more flexible and easier to change.

In [ ]:
class Engine:
    def start(self):
        return "engine on"

class Car:
    def __init__(self):
        self.engine = Engine()            # Car HAS an Engine

    def start(self):
        return f"car: {self.engine.start()}"

print(Car().start())

## 14.8 The template-method pattern

A base class defines the *skeleton* of a procedure; subclasses fill in the steps.

In [ ]:
class Pipeline:
    """Run a fixed sequence: load -> transform -> report."""

    def run(self, data):
        loaded = self.load(data)
        transformed = self.transform(loaded)
        return self.report(transformed)

    def load(self, data):
        return list(data)

    def transform(self, items):
        raise NotImplementedError

    def report(self, items):
        return f"{self.__class__.__name__}: {items}"

class Doubler(Pipeline):
    def transform(self, items):
        return [x * 2 for x in items]

class Cleaner(Pipeline):
    def transform(self, items):
        return [x.strip() for x in items]

print(Doubler().run([1, 2, 3]))
print(Cleaner().run(["  a ", " b"]))

## 14.9 Underscore conventions

| Name | Meaning |
|---|---|
| `_name` | internal use by convention |
| `__name` | name-mangled to `_Class__name` |
| `__name__` | special/dunder, defined by Python |
| `_` | throwaway variable |

In [ ]:
_hidden = 42
print(_hidden)

class Box:
    def __init__(self):
        self.__secret = 1
print(hasattr(Box(), "__secret"), hasattr(Box(), "_Box__secret"))   # False True

## Common pitfalls

- Forgetting `self` in a method definition or when accessing attributes.
- Defining `__eq__` without `__hash__` (the class becomes unhashable).
- Mutable class attributes (`items = []` at class level) shared by every instance.
- Deep inheritance chains; prefer shallow hierarchies and composition.
- Not calling `super().__init__()` in a subclass initialiser.

## Practice

1. Write a `Vector2D` class with `__add__`, `__eq__`, `__repr__`, and `__abs__`.
2. Build `Employee` and `Manager(Employee)` classes where `Manager` has a list of reports and overrides `describe()`.
3. Add a property `fahrenheit` to `Temperature` that converts on the fly.